# Address Number Coverage in Idealista Sale URLs

This notebook analyzes `data/idealista_barcelona_sale_urls.csv` to check whether `address_search` contains at least one digit, using that as a proxy for a more precise address. It also converts `price_search` into a numeric euro value and summarizes price differences by address-number status.

In [ ]:
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

In [ ]:
URLS_CSV = Path("data/idealista_barcelona_sale_urls.csv")
OUTPUT_CSV = Path("data/idealista_barcelona_sale_urls_address_number_analysis.csv")

df = pd.read_csv(URLS_CSV)
print(f"Loaded {len(df):,} rows from {URLS_CSV}")
df.head()

## Create Address-Number Flag and Numeric Price

In [ ]:
def has_number(value):
    if pd.isna(value):
        return False
    return bool(re.search(r"\d", str(value)))


def parse_price_eur(value):
    if pd.isna(value):
        return pd.NA
    text = str(value)
    # Remove currency symbols, spaces, thousands separators, and any trailing labels.
    match = re.search(r"\d[\d.,]*", text)
    if not match:
        return pd.NA
    number_text = match.group(0).replace(".", "").replace(",", "")
    try:
        return int(number_text)
    except ValueError:
        return pd.NA


df["address_search"] = df["address_search"].astype("string")
df["address_has_number"] = df["address_search"].apply(has_number)
df["address_number_status"] = df["address_has_number"].map({True: "has_number", False: "no_number"})
df["price_eur"] = df["price_search"].apply(parse_price_eur).astype("Int64")

df[["propertyCode", "address_search", "address_has_number", "price_search", "price_eur"]].head(10)

## Summary: Listings With vs Without Numbers in `address_search`

In [ ]:
address_number_summary = (
    df.groupby("address_number_status", dropna=False)
    .agg(
        listings=("propertyCode", "count"),
        listings_with_price=("price_eur", "count"),
        median_price_eur=("price_eur", "median"),
        mean_price_eur=("price_eur", "mean"),
    )
    .reset_index()
)
address_number_summary["share_of_all_listings"] = address_number_summary["listings"] / len(df)
address_number_summary["share_of_priced_listings"] = address_number_summary["listings_with_price"] / df["price_eur"].notna().sum()

display(address_number_summary)

print(f"Total listings: {len(df):,}")
print(f"Listings with price parsed: {df['price_eur'].notna().sum():,}")
print(f"Listings missing parsed price: {df['price_eur'].isna().sum():,}")

## Median Price by Address-Number Status

In [ ]:
median_price_by_number = (
    df.dropna(subset=["price_eur"])
    .groupby("address_number_status")
    .agg(
        listings=("propertyCode", "count"),
        median_price_eur=("price_eur", "median"),
        p25_price_eur=("price_eur", lambda x: x.quantile(0.25)),
        p75_price_eur=("price_eur", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

display(median_price_by_number)

## Address-Number Share by Price Bucket

In [ ]:
bucket_edges = [0, 250_000, 500_000, 750_000, 1_000_000, float("inf")]
bucket_labels = ["under_250k", "251_500k", "501_750k", "750k_1m", "1m_plus"]

df["price_bucket"] = pd.cut(
    df["price_eur"],
    bins=bucket_edges,
    labels=bucket_labels,
    include_lowest=True,
    right=True,
)

bucket_counts = (
    df.dropna(subset=["price_bucket"])
    .groupby(["price_bucket", "address_number_status"], observed=False)
    .size()
    .unstack(fill_value=0)
)

for col in ["has_number", "no_number"]:
    if col not in bucket_counts.columns:
        bucket_counts[col] = 0

bucket_counts = bucket_counts[["has_number", "no_number"]]
bucket_counts["total"] = bucket_counts.sum(axis=1)
bucket_counts["pct_has_number"] = bucket_counts["has_number"] / bucket_counts["total"]
bucket_counts["pct_no_number"] = bucket_counts["no_number"] / bucket_counts["total"]

display(bucket_counts.reset_index())

## Optional: Cleaner Display as Percentages

In [ ]:
bucket_display = bucket_counts.reset_index().copy()
bucket_display["pct_has_number"] = (bucket_display["pct_has_number"] * 100).round(1)
bucket_display["pct_no_number"] = (bucket_display["pct_no_number"] * 100).round(1)
bucket_display = bucket_display.rename(
    columns={
        "has_number": "n_has_number",
        "no_number": "n_no_number",
        "pct_has_number": "pct_has_number",
        "pct_no_number": "pct_no_number",
    }
)

display(bucket_display)

## Save Analysis Dataset

In [ ]:
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved row-level analysis dataset to {OUTPUT_CSV.resolve()}")